In [59]:
setwd("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/cortex")

source("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/code/enrichment_fxns.R")
source("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/code/gene_mapping_fxns.R")
source("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/code/top_corr_module_fxns.R")

Here I perform enrichment analysis to find modules enriched for cell type markers. 

These modules will later be used to correlate with exon PSI to define cell type-specific exons.

In [6]:
mod_def <- "PosBC"
unique <- FALSE

# Prep marker genes

### Claude marker genes

In [70]:
set_source <- "Claude_marker_genes"

marker_genes_list <- readRDS("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/marker_genes/AI/Claude_cortical_markers_human.RDS")

### MOp pseudobulk DE genes

In [ ]:
set_source <- "yao_2021_MOp_STAR_donor_subclass_label_pseudobulk_pairwise_DE_genes"

pairwise_res_list <- readRDS("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/pseudobulk_test/yao_2021/MOp/data/DE_genes/yao_2021_MOp_STAR_donor_subclass_label_pseudobulk_pairwise_DE_genes_dream.RDS")
marker_genes_list <- prep_DE_genes(pairwise_res_list, pairwise=TRUE, unique=unique)
names(marker_genes_list) <- unlist(sapply(names(marker_genes_list), function(x) gsub(" ", "_", x)))

# Convert to human markers
marker_genes_list <- lapply(marker_genes_list, function(gene_list) {
    convert_genes(gene_list, target_species="human")
})

### ACA pseudobulk DE genes

In [63]:
set_source <- "yao_2021_ACA_STAR_donor_subclass_label_pseudobulk_pairwise_DE_genes"

pairwise_res_list <- readRDS("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/pseudobulk_test/yao_2021/ACA/data/DE_genes/yao_2021_ACA_STAR_donor_subclass_label_pseudobulk_pairwise_DE_genes_dream.RDS")
marker_genes_list <- prep_DE_genes(pairwise_res_list, lfc_threshold, pairwise=TRUE, unique=unique)

# Convert to human markers
marker_genes_list <- lapply(marker_genes_list, function(gene_list) {
    convert_genes(gene_list, target_species="human")
})

### MO marker genes

In [3]:
# load("/mnt/lareaulab/reliscu/data/gene_sets/Oldham/MO_sets.RData")

In [4]:
# sets <- c(
#     "BARRES_ASTROCYTES",
#     "BARRES_OLIGODENDROCYTES",
#     "HICKMAN_MICROGLIA_SENSOME_99",
#     "BARRES_NEURONS",
#     "BUTLER_ENDOTHELIAL_HPA",
#     "ZEISEL_MURAL",
#     "ZEISEL_EPENDYMAL",
#     "ZEISEL_INTERNEURON",
#     "ZHANG_MOUSE_OPC_2014",
#     "Mukamel_InhibitoryNeurons",
#     "Mukamel_ExcitatoryNeurons",
#     "SUGINO_UP_GABAERGIC_NEURONS",
#     "SUGINO_UP_GLUTAMATERGIC_NEURONS",
#     "SUGINO_UP_LAYERS4-6_GABAERGIC_NEURONS_CINGULATE_CTX",
#     "BAKKEN_2019_PVALB_GABAERGIC_DE_GABA_CLUSTERS",
#     "BAKKEN_2019_VIP_GABAERGIC_DE_GABA_CLUSTERS",
#     "BAKKEN_2019_SST_GABAERGIC_DE_GABA_CLUSTERS",
#     "BAKKEN_2019_LAMP5_GABAERGIC_DE_GABA_CLUSTERS",
#     "BAKKEN_2019_SNCG_GABAERGIC_DE_GABA_CLUSTERS",
#     "BAKKEN_2019_SST_CHODL_GABAERGIC_DE_GABA_CLUSTERS",
#     "VELMESHEV_2019_IN_SST",
#     MO_legend$SetName[grepl("schirmer", MO_legend$SetName, ignore.case=T)],
#     MO_legend$SetName[grepl("yang_pfc", MO_legend$SetName, ignore.case=T)]
# )

In [5]:
# mask <- MO_legend$SetName %in% sets

# marker_genes_list <- MO_sets[mask]
# names(marker_genes_list) <- MO_legend$SetName[mask] 

# marker_genes_list <- lapply(marker_genes_list, toupper)

### Gugene DE genes

In [4]:
marker_genes_list <- readRDS("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/kang_2026_preprint/kang_2026_preprint_DE_genes.RData")

In [ ]:
marker_genes_list1 <- readRDS("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/kang_2026_preprint/kang_2026_preprint_Jorstad_2023_DE_genes.RData")
marker_genes_list2 <- readRDS("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/kang_2026_preprint/kang_2026_preprint_Gabitto_2024_DE_genes.RData")

marker_genes_list <- lapply(seq_along(marker_genes_list1), function(i) {
    union(marker_genes_list1[[i]], marker_genes_list2[[i]])
})

names(marker_genes_list) <- names(marker_genes_list1)

# Get enrichments

### TPM data

In [51]:
network_dir <- "GTEx_cortex_TPM_All_598_outliers_removed_ComBat_SMGEBTCH_corrected_mergeParam0.95_subsetCutoff0.26_Modules"

In [71]:
enrichments_df <- get_module_enrichments(network_dir, marker_genes_list, mod_def)

In [74]:
# Get most enriched cell type for each module
# If cell type is most enriched in multiple modules, choose module with smallest p-value

top_mods_df <- enrichments_df %>%
    group_by(Cell_type) %>%
    slice_min(Qval, with_ties=FALSE) %>%
    group_by(Network, Module) %>%
    arrange(Network) %>%
    slice_min(Qval, with_ties=FALSE) %>%
    arrange(Qval)

In [75]:
top_mods_df[,c("Cell_type", "Pval", "Qval", "Module", "Network")]

Cell_type,Pval,Qval,Module,Network
<chr>,<dbl>,<dbl>,<chr>,<chr>
Pvalb,1.653881e-27,4.258743e-23,plum2,Bicor-None_signum0.8_minSize8_merge_ME_0.95_22277
Sst,6.467972e-27,5.102260e-23,plum2,Bicor-None_signum0.8_minSize10_merge_ME_0.95_22277
Sst Chodl,6.747673e-25,1.241090e-21,darkgreen,Bicor-None_signum0.843_minSize6_merge_ME_0.95_22277
Vip,2.109779e-24,3.195694e-21,lightcyan,Bicor-None_signum0.843_minSize8_merge_ME_0.95_22277
Micro/PVM,4.172165e-24,5.654381e-21,skyblue,Bicor-None_signum0.8_minSize6_merge_ME_0.95_22277
Pax6,2.391264e-19,9.773818e-17,greenyellow,Bicor-None_signum0.843_minSize10_merge_ME_0.95_22277
Oligo,6.058142e-19,2.197143e-16,tan,Bicor-None_signum0.8_minSize10_merge_ME_0.95_22277
Astro,7.173324e-19,2.530316e-16,mediumorchid,Bicor-None_signum0.82_minSize6_merge_ME_0.95_22277
VLMC,1.183773e-12,1.870070e-10,white,Bicor-None_signum0.8_minSize8_merge_ME_0.95_22277


In [76]:
get_mod_vs_DE_genes(top_mods_df, marker_genes_list, mod_def)

,Cell_type,Qval,Module_genes,DE_genes,Intersection_genes,Module,Network,ME_path,kME_path
,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
Pvalb,Pvalb,4.258743e-23,"ZNF385D, GAD2, GAD1, MYO5B, LGI2, KCNS3, NXPH2, VWC2, BTBD11, NXPH1, TACR1, LHX6, SLC32A1, MINAR1, ARL4C","GAD1, GAD2, LHX6, PVALB, SLC32A1, DLX1, DLX2, DLX5, ERBB4, KCNC1, MYO5B, NKX2-1, RBFOX3, SOX6, SYT2","GAD1,GAD2,LHX6,SLC32A1,DLX1,ERBB4,MYO5B,TAC1,DLX6,KCNC2,MAP2,PTHLH,STMN2",plum2,Bicor-None_signum0.8_minSize8_merge_ME_0.95_22277,/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/cortex/GTEx_cortex_TPM_All_598_outliers_removed_ComBat_SMGEBTCH_corrected_mergeParam0.95_subsetCutoff0.26_Modules/Bicor-None_signum0.8_minSize8_merge_ME_0.95_22277/Module_eigengenes_01-44-19.csv,/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/cortex/GTEx_cortex_TPM_All_598_outliers_removed_ComBat_SMGEBTCH_corrected_mergeParam0.95_subsetCutoff0.26_Modules/Bicor-None_signum0.8_minSize8_merge_ME_0.95_22277/kME_table_01-44-19.csv
Sst,Sst,5.102260e-23,"ZNF385D, GAD2, GAD1, MYO5B, LGI2, KCNS3, NXPH2, VWC2, BTBD11, NXPH1, TACR1, LHX6, SLC32A1, MINAR1, ARL4C","GAD1, GAD2, LHX6, SLC32A1, SST, CALB1, CALB2, CBLN4, CHRNA2, DLX1, DLX2, DLX5, ETV1, HPSE, MYH8","GAD1,GAD2,LHX6,SLC32A1,DLX1,DLX6,MAFB,MAP2,STMN2",plum2,Bicor-None_signum0.8_minSize10_merge_ME_0.95_22277,/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/cortex/GTEx_cortex_TPM_All_598_outliers_removed_ComBat_SMGEBTCH_corrected_mergeParam0.95_subsetCutoff0.26_Modules/Bicor-None_signum0.8_minSize10_merge_ME_0.95_22277/Module_eigengenes_02-05-47.csv,/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/cortex/GTEx_cortex_TPM_All_598_outliers_removed_ComBat_SMGEBTCH_corrected_mergeParam0.95_subsetCutoff0.26_Modules/Bicor-None_signum0.8_minSize10_merge_ME_0.95_22277/kME_table_02-05-47.csv
Sst Chodl,Sst Chodl,1.241090e-21,"ZNF385D, GAD2, GAD1, MYO5B, LGI2, KCNS3, NXPH2, VWC2, BTBD11, NXPH1, TACR1, LHX6, SLC32A1, MINAR1, ARL4C","CHODL, GAD1, GAD2, LHX6, SLC32A1, SST, DLX1, DLX2, DLX5, NKX2-1, NOS1, NPY, RBFOX3, SOX6, TACR1","GAD1,GAD2,LHX6,SLC32A1,DLX1,TACR1,DLX6,MAFB,MAP2,STMN2",darkgreen,Bicor-None_signum0.843_minSize6_merge_ME_0.95_22277,/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/cortex/GTEx_cortex_TPM_All_598_outliers_removed_ComBat_SMGEBTCH_corrected_mergeParam0.95_subsetCutoff0.26_Modules/Bicor-None_signum0.843_minSize6_merge_ME_0.95_22277/Module_eigengenes_11-59-44.csv,/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/cortex/GTEx_cortex_TPM_All_598_outliers_removed_ComBat_SMGEBTCH_corrected_mergeParam0.95_subsetCutoff0.26_Modules/Bicor-None_signum0.843_minSize6_merge_ME_0.95_22277/kME_table_11-59-44.csv
Vip,Vip,3.195694e-21,"ZNF385D, GAD2, GAD1, MYO5B, LGI2, KCNS3, NXPH2, VWC2, BTBD11, NXPH1, TACR1, LHX6, SLC32A1, MINAR1, ARL4C","ADARB2, GAD1, GAD2, SLC32A1, VIP, CALB2, CHAT, CRH, DLX1, DLX2, DLX5, GPC3, HTR3A, MYBPC1, NR2F2","GAD1,GAD2,SLC32A1,DLX1,DLX6,MAP2,STMN2",lightcyan,Bicor-None_signum0.843_minSize8_merge_ME_0.95_22277,/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/cortex/GTEx_cortex_TPM_All_598_outliers_removed_ComBat_SMGEBTCH_corrected_mergeParam0.95_subsetCutoff0.26_Modules/Bicor-None_signum0.843_minSize8_merge_ME_0.95_22277/Module_eigengenes_12-20-19.csv,/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/cortex/GTEx_cortex_TPM_All_598_outliers_removed_ComBat_SMGEBTCH_corrected_mergeParam0.95_subsetCutoff0.26_Modules/Bicor-None_signum0.843_minSize8_merge_ME_0.95_22277/kME_table_12-20-19.csv
Micro/PVM,Micro/PVM,5.654381e-21,"ITGB2, C1QC, C1QA, LAIR1, C1QB, SPI1, LAPTM5, CD300A, SASH3, TBXAS1, NCKAP1L, LAT2, HCK, TYROBP, ALOX5AP","CTSS, CX3CR1, ITGAM, P2RY12, TMEM119, TYROBP, AIF1, C1QA, C1QB, C1QC, CSF1R, HEXB, SALL1, SPI1, TREM2","CTSS,CX3CR1,ITGAM,TMEM119,TYROBP,AIF1,C1QA,C1QB,C1QC,CSF1R,SPI1,TREM2",skyblue,Bicor-None_signum0.8_minSize6_merge_ME_0.95_22277,/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/cortex/GTEx_cortex

In [77]:
write.csv(top_mods_df, file=paste0("data/enrichments/", network_dir, "_", set_source, "_enrichments.csv"), row.names=FALSE, quote=FALSE)

### Counts data

In [26]:
network_dir <- "GTEx_cortex_counts_rd2_All_501_outliers_removed_ComBat_SMGEBTCH_corrected_mergeParam0.95_subsetCutoff0.002_Modules"

In [ ]:
enrichments_df <- get_module_enrichments(network_dir, marker_genes_list, mod_def)

In [ ]:
# Get most enriched cell type for each module
# If cell type is most enriched in multiple modules, choose module with smallest p-value

top_mods_df <- enrichments_df %>%
    group_by(Cell_type) %>%
    slice_min(Qval, with_ties=FALSE) %>%
    group_by(Network, Module) %>%
    arrange(Network) %>%
    slice_min(Qval, with_ties=FALSE) %>%
    arrange(Qval)

In [ ]:
top_mods_df[,c("Cell_type", "Pval", "Qval", "Module", "Network")]

Cell_type,Pval,Qval,Module,Network
<chr>,<dbl>,<dbl>,<chr>,<chr>
Astro,4.432476e-112,3.892423e-107,green,Bicor-None_signum0.898_minSize6_merge_ME_0.95_22174
Micro/PVM,4.465014e-85,1.960498e-80,ivory,Bicor-None_signum0.898_minSize3_merge_ME_0.95_22174
Oligo,2.508114e-73,3.670876e-69,pink,Bicor-None_signum0.82_minSize10_merge_ME_0.95_22174
Endo,9.136950e-37,3.086040e-33,lightgoldenrodyellow,Bicor-None_signum0.665_minSize8_merge_ME_0.95_22174
VLMC,1.723223e-28,4.323616e-25,antiquewhite1,Bicor-None_signum0.665_minSize10_merge_ME_0.95_22174
OPC,3.363768e-16,3.009703e-13,grey60,Bicor-None_signum0.82_minSize8_merge_ME_0.95_22174
L2/3 IT,9.579135e-13,6.421384e-10,plum1,Bicor-None_signum0.82_minSize6_merge_ME_0.95_22174
L4 IT,4.493806e-10,2.110310e-07,darkgreen,Bicor-None_signum0.82_minSize8_merge_ME_0.95_22174
L5 ET,7.021306e-09,2.680796e-06,goldenrod1,Bicor-None_signum0.898_minSize2_merge_ME_0.95_22174


In [ ]:
write.csv(top_mods_df, file=paste0("data/enrichments/", network_dir, "_", set_source, "_enrichments.csv"), row.names=FALSE, quote=FALSE)